In [2]:
import os
from pinecone import Pinecone
from langchain_community.retrievers import PineconeHybridSearchRetriever
# from sentence_transformers import SentenceTransformer
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder

# from langchain.chat_models import init_chat_model
from langchain_core.messages import  HumanMessage, SystemMessage

/home/xeeteex/ocr-extraction/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Cost for Langchain SDK usage

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from google.oauth2 import service_account
from backend.core.config import settings

credentials = service_account.Credentials.from_service_account_file(
    settings.GOOGLE_APPLICATION_CREDENTIALS,
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    credentials=credentials,
    project= settings.GOOGLE_CLOUD_PROJECT.get_secret_value(),
    location=settings.GOOGLE_CLOUD_LOCATION,
    # Uses Application Default Credentials (ADC)
)

response= llm.invoke("2+2")  # should return "4"
print(response)

input_tokens = response.usage_metadata.get("input_tokens", None)
print(input_tokens) # number of input tokens



content='2 + 2 = 4' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019bf9e3-7fdd-7971-9ad7-74444cafbb94-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 3, 'output_tokens': 61, 'total_tokens': 64, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 54}}


In [26]:
input_tokens = response.usage_metadata.get("input_tokens", None)
print(input_tokens) # number of input tokens

3


In [28]:
output_tokens = response.usage_metadata.get("output_tokens", None)
print(output_tokens) # number of output tokens

61


 # Cost for SDK usage

In [ ]:
from google import genai
from backend.core.config import settings

model= genai.Client(
    vertexai=True,  
    project=settings.GOOGLE_CLOUD_PROJECT.get_secret_value(),
    location=settings.GOOGLE_CLOUD_LOCATION,
    
)   

response = model.models.generate_content(
            model="gemini-2.5-flash",
            contents = "2+2",
            # config=genai.types.GenerateContentConfig(
            #     # response_mime_type="application/json",
            #     temperature=0.1
            # )
        )


print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=10>
) candidates=[Candidate(
  avg_logprobs=-13.563299179077148,
  content=Content(
    parts=[
      Part(
        text='4'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>
)] create_time=datetime.datetime(2026, 1, 26, 11, 21, 27, 920987, tzinfo=TzInfo(0)) model_version='gemini-2.5-flash' prompt_feedback=None response_id='N053aZubOLSIm9IPiJP1oQE' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=1,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=1
    ),
  ],
  prompt_token_count=3,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=3
    ),
  ],
  thoughts_token_count=23,
  total_token_count=27,
  traffic_type=<TrafficType.ON_DEMAND: 'ON_DEMAND'>
) automatic_function_calling_history=[] parsed=None


In [ ]:
input_tokens = response.usage_metadata.prompt_token_count
print(input_tokens) # number of input tokens
# for gemini 2.5 lite For Input (text, image, video)	$0.1 per 1M tokens
# for gemini 2.5 flash For Input (text, image, video)	$0.3 per 1M tokens

input_tokens_price = (input_tokens / 1000000) * 0.1 
print(f"{input_tokens_price:.8f}")

3
0.00000030


In [ ]:
candidates_tokens = response.usage_metadata.candidates_token_count
thoughts_tokens = response.usage_metadata.thoughts_token_count
total_output_tokens = candidates_tokens + thoughts_tokens
print(total_output_tokens) # number of output tokens
# for gemini 2.5 lite For Output (text, image, video)	$0.4 per 1M tokens
# for gemini 2.5 flash For Output (text, image, video)	$2.5 per 1M tokens

output_tokens_price = (total_output_tokens / 1000000) * 0.4 
print(f"{output_tokens_price:.8f}")

24
0.00000960


### Logic per model of Gemini

# Hybrid Retriever

In [10]:
from pinecone import Pinecone
from langchain_community.retrievers import PineconeHybridSearchRetriever
# from sentence_transformers import SentenceTransformer
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder

In [ ]:
from backend.core.config import settings
embeddings = HuggingFaceEmbeddings(model_name=settings.EMBEDDING_MODEL)

In [12]:
bm25_encoder = BM25Encoder().default()

In [ ]:
from backend.core.config import settings

pc = Pinecone(settings.PINECONE_API_KEY)
index = pc.Index(settings.PINECONE_INDEX_NAME) 
namespaces = ["raw_material", "auxiliary_raw_material", "electrical"]

total_fetched_result = []

for namespace in namespaces:
    query = "gl side slit"
    retriever = PineconeHybridSearchRetriever(
                        embeddings= embeddings,
                        sparse_encoder=bm25_encoder,
                        index=index,
                        namespace=namespace,
                        top_k= 25,
                        alpha=0.1,
                        text_key="text",
                    )
    result = retriever.invoke(query)
    print(result)

    for doc in result:
        fetched_result = [doc.page_content]
        total_fetched_result.extend(fetched_result)
        

print(len(total_fetched_result))
print(total_fetched_result)

[Document(metadata={'InventoryUoMEntry': 1, 'ItemCode': 'RM02792', 'UoMGroupEntry': 1, 'score': 0.0244930983}, page_content='MS WIRE 8 MM'), Document(metadata={'InventoryUoMEntry': 1, 'ItemCode': 'RM2795', 'UoMGroupEntry': 1, 'score': 0.0241150856}, page_content='MS WIRE 5.50 MM'), Document(metadata={'InventoryUoMEntry': 1, 'ItemCode': 'RM02786', 'UoMGroupEntry': 1, 'score': 0.0229854584}, page_content='Light Scrap'), Document(metadata={'InventoryUoMEntry': 1, 'ItemCode': 'RM02783', 'UoMGroupEntry': 1, 'score': 0.0191881657}, page_content='Dehati Scrap'), Document(metadata={'InventoryUoMEntry': 1, 'ItemCode': 'RM02787', 'UoMGroupEntry': 1, 'score': 0.0190014839}, page_content='Light/Tina/Burada'), Document(metadata={'InventoryUoMEntry': 1, 'ItemCode': 'RM02781', 'UoMGroupEntry': 1, 'score': 0.0163851976}, page_content='Burada Scrap'), Document(metadata={'InventoryUoMEntry': 10, 'ItemCode': 'RM02794', 'UoMGroupEntry': 9, 'score': 0.00866216421}, page_content='ENAMEL RED PAINT'), Documen